# Part B: WGAN and WGAN-GP

CIFAR-10 image generation assignment - Part B.

Trains a DCGAN-style generator/critic pair two ways:
- **WGAN**: Wasserstein loss with weight clipping on the critic, RMSprop optimizer
- **WGAN-GP**: Wasserstein loss with a gradient-penalty term instead of clipping, Adam optimizer

Both are trained on the same CIFAR-10 subset, generate random samples, and are scored with FID.

**Before running:** in Colab, go to `Runtime > Change runtime type` and select a GPU.

This notebook is self-contained (does not depend on Part A). To keep Colab GPU time
reasonable it trains on a 15,000-image subset for a reduced number of epochs; increase
`SUBSET_SIZE` / `EPOCHS_*` for a more rigorous final run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || print("No GPU detected - go to Runtime > Change runtime type > GPU")

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/samples", exist_ok=True)

## 1. Data: CIFAR-10, normalized to [0,1]

In [ ]:
SUBSET_SIZE = 15000
TEST_SUBSET_SIZE = 3000
BATCH_SIZE = 64

transform = T.Compose([T.ToTensor()])  # already scales to [0,1]

train_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

g = torch.Generator().manual_seed(SEED)
train_idx = torch.randperm(len(train_full), generator=g)[:SUBSET_SIZE]
test_idx = torch.randperm(len(test_full), generator=g)[:TEST_SUBSET_SIZE]

train_set = Subset(train_full, train_idx)
test_set = Subset(test_full, test_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train subset: {len(train_set)} images, Test subset: {len(test_set)} images")

## 2. Evaluation utility: FID

Same recipe as Part A - ImageNet-pretrained InceptionV3 pool features, Frechet distance
between Gaussian fits to real vs. generated activations. Duplicated here so this notebook
runs standalone.

In [ ]:
from scipy import linalg
import torchvision.models as tvm

class InceptionFeatureExtractor(nn.Module):
    def __init__(self, device):
        super().__init__()
        weights = tvm.Inception_V3_Weights.IMAGENET1K_V1
        net = tvm.inception_v3(weights=weights, aux_logits=True)
        net.fc = nn.Identity()
        net.eval()
        for p in net.parameters():
            p.requires_grad_(False)
        self.net = net.to(device)
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.no_grad()
    def features(self, x):
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return self.net(x).detach().cpu().numpy()

_fid_extractor = None
def get_fid_extractor():
    global _fid_extractor
    if _fid_extractor is None:
        _fid_extractor = InceptionFeatureExtractor(device)
    return _fid_extractor

@torch.no_grad()
def extract_features(images, batch_size=64):
    extractor = get_fid_extractor()
    feats = []
    for i in range(0, images.size(0), batch_size):
        batch = images[i:i + batch_size].to(device)
        feats.append(extractor.features(batch))
    return np.concatenate(feats, axis=0)

def frechet_distance(feat_real, feat_fake, eps=1e-6):
    mu1, mu2 = feat_real.mean(0), feat_fake.mean(0)
    sigma1 = np.cov(feat_real, rowvar=False)
    sigma2 = np.cov(feat_fake, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))

def compute_fid(real_images, fake_images):
    return frechet_distance(extract_features(real_images), extract_features(fake_images))

FID_REF_N = 2000
_real_ref_imgs = torch.cat([b for b, _ in test_loader], dim=0)[:FID_REF_N]
print("FID reference set:", _real_ref_imgs.shape)

## 3. Models: DCGAN-style Generator and Critic

Same architecture is reused for WGAN and WGAN-GP; only the training objective and
optimizer differ. The critic has no sigmoid output (Wasserstein critics output an
unbounded real-valued score, not a probability).

In [ ]:
LATENT_DIM = 128

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),  # 4x4
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),          # 8x8
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),            # 16x16
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Sigmoid(),                                            # 32x32, in [0,1]
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.InstanceNorm2d(64, affine=True), nn.LeakyReLU(0.2, inplace=True),   # 16x16
            nn.Conv2d(64, 128, 4, 2, 1), nn.InstanceNorm2d(128, affine=True), nn.LeakyReLU(0.2, inplace=True),# 8x8
            nn.Conv2d(128, 256, 4, 2, 1), nn.InstanceNorm2d(256, affine=True), nn.LeakyReLU(0.2, inplace=True),# 4x4
            nn.Conv2d(256, 1, 4, 1, 0),                                                                        # 1x1
        )

    def forward(self, x):
        return self.net(x).view(-1)

def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 4. WGAN (weight clipping)

In [ ]:
EPOCHS_WGAN = 40
N_CRITIC = 5
CLIP_VALUE = 0.01
LR_WGAN = 5e-5

netG_wgan = Generator().to(device); netG_wgan.apply(weights_init)
netD_wgan = Critic().to(device); netD_wgan.apply(weights_init)

optG_wgan = torch.optim.RMSprop(netG_wgan.parameters(), lr=LR_WGAN)
optD_wgan = torch.optim.RMSprop(netD_wgan.parameters(), lr=LR_WGAN)

fixed_noise = torch.randn(64, LATENT_DIM, device=device)
wgan_history = {"d_loss": [], "g_loss": []}

step = 0
for epoch in range(1, EPOCHS_WGAN + 1):
    d_losses, g_losses = [], []
    for real, _ in train_loader:
        real = real.to(device)
        b = real.size(0)

        # --- critic step(s) ---
        for _ in range(N_CRITIC):
            z = torch.randn(b, LATENT_DIM, device=device)
            with torch.no_grad():
                fake = netG_wgan(z)
            d_real = netD_wgan(real).mean()
            d_fake = netD_wgan(fake).mean()
            d_loss = -(d_real - d_fake)  # maximize (d_real - d_fake) == minimize its negative

            optD_wgan.zero_grad()
            d_loss.backward()
            optD_wgan.step()
            for p in netD_wgan.parameters():
                p.data.clamp_(-CLIP_VALUE, CLIP_VALUE)

        # --- generator step ---
        z = torch.randn(b, LATENT_DIM, device=device)
        fake = netG_wgan(z)
        g_loss = -netD_wgan(fake).mean()

        optG_wgan.zero_grad()
        g_loss.backward()
        optG_wgan.step()

        d_losses.append(d_loss.item()); g_losses.append(g_loss.item())
        step += 1

    wgan_history["d_loss"].append(float(np.mean(d_losses)))
    wgan_history["g_loss"].append(float(np.mean(g_losses)))
    print(f"[WGAN] epoch {epoch:02d}/{EPOCHS_WGAN}  D_loss={wgan_history['d_loss'][-1]:.4f}  G_loss={wgan_history['g_loss'][-1]:.4f}")

    if epoch % 10 == 0 or epoch == EPOCHS_WGAN:
        with torch.no_grad():
            samples = netG_wgan(fixed_noise).cpu()
        grid = vutils.make_grid(samples, nrow=8)
        plt.figure(figsize=(8, 8)); plt.axis("off"); plt.title(f"WGAN samples, epoch {epoch}")
        plt.imshow(grid.permute(1, 2, 0).numpy()); plt.show()

## 5. WGAN-GP (gradient penalty)

In [ ]:
EPOCHS_WGANGP = 40
N_CRITIC_GP = 5
LAMBDA_GP = 10.0
LR_WGANGP = 1e-4
BETAS_ADAM = (0.5, 0.9)

netG_gp = Generator().to(device); netG_gp.apply(weights_init)
netD_gp = Critic().to(device); netD_gp.apply(weights_init)

optG_gp = torch.optim.Adam(netG_gp.parameters(), lr=LR_WGANGP, betas=BETAS_ADAM)
optD_gp = torch.optim.Adam(netD_gp.parameters(), lr=LR_WGANGP, betas=BETAS_ADAM)

def gradient_penalty(critic, real, fake, device):
    b = real.size(0)
    eps = torch.rand(b, 1, 1, 1, device=device).expand_as(real)
    interpolates = (eps * real + (1 - eps) * fake).requires_grad_(True)
    d_interpolates = critic(interpolates)
    grads = autograd.grad(
        outputs=d_interpolates, inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grads = grads.view(b, -1)
    gp = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp

fixed_noise_gp = torch.randn(64, LATENT_DIM, device=device)
wgangp_history = {"d_loss": [], "g_loss": []}

for epoch in range(1, EPOCHS_WGANGP + 1):
    d_losses, g_losses = [], []
    for real, _ in train_loader:
        real = real.to(device)
        b = real.size(0)

        for _ in range(N_CRITIC_GP):
            z = torch.randn(b, LATENT_DIM, device=device)
            with torch.no_grad():
                fake = netG_gp(z)
            d_real = netD_gp(real).mean()
            d_fake = netD_gp(fake).mean()
            gp = gradient_penalty(netD_gp, real, fake, device)
            d_loss = -(d_real - d_fake) + LAMBDA_GP * gp

            optD_gp.zero_grad()
            d_loss.backward()
            optD_gp.step()

        z = torch.randn(b, LATENT_DIM, device=device)
        fake = netG_gp(z)
        g_loss = -netD_gp(fake).mean()

        optG_gp.zero_grad()
        g_loss.backward()
        optG_gp.step()

        d_losses.append(d_loss.item()); g_losses.append(g_loss.item())

    wgangp_history["d_loss"].append(float(np.mean(d_losses)))
    wgangp_history["g_loss"].append(float(np.mean(g_losses)))
    print(f"[WGAN-GP] epoch {epoch:02d}/{EPOCHS_WGANGP}  D_loss={wgangp_history['d_loss'][-1]:.4f}  G_loss={wgangp_history['g_loss'][-1]:.4f}")

    if epoch % 10 == 0 or epoch == EPOCHS_WGANGP:
        with torch.no_grad():
            samples = netG_gp(fixed_noise_gp).cpu()
        grid = vutils.make_grid(samples, nrow=8)
        plt.figure(figsize=(8, 8)); plt.axis("off"); plt.title(f"WGAN-GP samples, epoch {epoch}")
        plt.imshow(grid.permute(1, 2, 0).numpy()); plt.show()

## 6. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(wgan_history["d_loss"], label="D loss")
axes[0].plot(wgan_history["g_loss"], label="G loss")
axes[0].set_title("WGAN training loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(wgangp_history["d_loss"], label="D loss")
axes[1].plot(wgangp_history["g_loss"], label="G loss")
axes[1].set_title("WGAN-GP training loss"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/gan_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Generate random samples and compute FID

In [ ]:
N_GEN = 2000  # for a stable FID estimate
N_SHOW = 100  # for Part C visual comparison

with torch.no_grad():
    z = torch.randn(N_GEN, LATENT_DIM, device=device)
    wgan_samples = torch.cat([netG_wgan(z[i:i+256]).cpu() for i in range(0, N_GEN, 256)], dim=0)

    z = torch.randn(N_GEN, LATENT_DIM, device=device)
    wgangp_samples = torch.cat([netG_gp(z[i:i+256]).cpu() for i in range(0, N_GEN, 256)], dim=0)

fid_wgan = compute_fid(_real_ref_imgs, wgan_samples)
fid_wgangp = compute_fid(_real_ref_imgs, wgangp_samples)

print(f"WGAN     FID = {fid_wgan:.2f}")
print(f"WGAN-GP  FID = {fid_wgangp:.2f}")

vutils.save_image(wgan_samples[:N_SHOW], "outputs/samples/wgan_100_samples.png", nrow=10)
vutils.save_image(wgangp_samples[:N_SHOW], "outputs/samples/wgangp_100_samples.png", nrow=10)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].bar(["WGAN", "WGAN-GP"], [fid_wgan, fid_wgangp], color=["indianred", "seagreen"])
axes[0].set_title("FID (lower is better)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("outputs/gan_fid_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save artifacts for Part C

In [ ]:
import json

torch.save(netG_wgan.state_dict(), "outputs/wgan_generator.pt")
torch.save(netD_wgan.state_dict(), "outputs/wgan_critic.pt")
torch.save(netG_gp.state_dict(), "outputs/wgangp_generator.pt")
torch.save(netD_gp.state_dict(), "outputs/wgangp_critic.pt")

metrics = {
    "wgan": {"fid": fid_wgan, "d_loss_history": wgan_history["d_loss"], "g_loss_history": wgan_history["g_loss"]},
    "wgangp": {"fid": fid_wgangp, "d_loss_history": wgangp_history["d_loss"], "g_loss_history": wgangp_history["g_loss"]},
}
with open("outputs/part_b_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved checkpoints, metrics, and Part C sample grids under outputs/")

!zip -rq outputs_part_b.zip outputs
print("Zipped -> outputs_part_b.zip (download this from the Colab file browser)")

## 9. Export to HTML (deliverable)

Run this in Colab after all cells above have executed (and after saving the notebook).

In [ ]:
NOTEBOOK_NAME = "Part_B_WGAN.ipynb"  # update if you renamed the file in Colab
!jupyter nbconvert --to html "{NOTEBOOK_NAME}" 2>/dev/null || print("Save the notebook first (Ctrl+S / Cmd+S), then re-run this cell.")